# LHCb $D_s^+\to\pi^-\pi^+\pi^+$ — real-data fit with COW sWeights

This example implements the nominal $D_s^+\to\pi^-\pi^+\pi^+$ amplitude model using the public Jax-PWA APIs. The particle ordering is $(\pi^-,\pi^+,\pi^+)$: `s12` and `s13` are the two opposite-sign $\pi^-\pi^+$ combinations and `s23` is the same-sign $\pi^+\pi^+$ combination.

The full candidate mass spectrum is first fitted with an **extended unbinned likelihood**, following Chapter 4 of LHCb-ANA-2022-011. The signal model is
$f_{CB}\,\mathrm{CrystalBall} + (1-f_{CB})\,\mathrm{Gaussian}$ with the original parameter relations
$\mu_G=\mu_{CB}+\mu_{\mathrm{offset}}$ and
$\sigma_G=\sigma_{CB}\,\sigma_{\mathrm{Ratio}}$; the combinatorial background is exponential. The fitted mass PDFs are then passed to the `sweights` package's stable `Cow` implementation to compute COW signal weights.

The Dalitz fit contains no explicit background histogram: it uses `WeightedUnbinnedNLL` with the COW signal weights and the Jax-PWA squared-weight covariance correction. The Dalitz variables remain the DTF/mass-constrained invariants so all candidates in the full mass-fit range share the nominal $D_s$ Dalitz boundary.

Reference amplitude model: LHCb, JHEP 07 (2023) 204, arXiv:2209.09840.


In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedUnbinnedNLL
from scipy.stats import crystalball, norm
from sweights import Cow

from jaxpwa import (
    DecayChannel,
    DecayModel,
    FitSession,
    GounarisSakurai,
    GooFitLegacyAngular,
    Parameter,
    PhaseSpaceSample,
    QMI,
    RealImag,
    Resonance,
    WeightedUnbinnedNLL,
    enable_x64,
    generate_toy,
    histogram_efficiency_from_root,
    read_root_tree,
)

enable_x64()
print('JAX backend:', jax.default_backend())
print('JAX devices:', jax.devices())


## 1. Input files, mass branch and Dalitz convention

Install the optional sPlot/COW dependency with `python -m pip install -e ".[splot]"`; the package already declares `splot = ["sweights"]` in `pyproject.toml`.

The ROOT branches are read in one call so the mass values and Dalitz coordinates stay event-by-event aligned. `MASS_TO_GEV` is explicit rather than inferred: keep `1e-3` for a mass branch stored in MeV and change it to `1.0` if the branch is already in GeV.


In [ ]:
DATA_FILE = Path('/home/juan-leite/Work/data/DsPPP_SignalRegion_AllSelectionCuts_Splot_FReweighted_MeanMVA_95.root')
DATA_TREE = 'DecayTree'

S12_BRANCH = 's12_pipi_DTF'
S13_BRANCH = 's13_pipi_DTF'
S23_BRANCH = 's23_pipi_DTF'
MASS_BRANCH = 'Ds_M'  # change to the mass branch in the full-spectrum sample
MASS_TO_GEV = 1e-3    # use 1.0 if MASS_BRANCH is already in GeV

# Chapter 4 / original sPlot fit range: 1910--2030 MeV.
MASS_RANGE_GEV = (1.910, 2.030)

DATA_CUT = None
ENTRY_START, ENTRY_STOP = None, None

EFFICIENCY_FILE = Path('/home/juan-leite/Work/data/acc_hist_15_95_Smoothed_normalized.root')
EFFICIENCY_HIST = 'h_eff'

RUN_PREFIT_TOY_COMPARISON = True
TOY_SEED = 20230922
RUN_FIT = True


## 2. Published QMI S-wave starting point

The 50 knots below are the central values used as the fit starting point. `QMI(..., interpolation='linear')` linearly interpolates the magnitude and phase parameters in $s=m^2$.

In [ ]:
QMI_KNOTS_GEV = np.array([
    0.280,0.390,0.470,0.546,0.623,0.698,0.766,0.819,0.865,0.900,
    0.925,0.942,0.955,0.964,0.972,0.978,0.983,0.990,1.001,1.023,
    1.051,1.083,1.116,1.149,1.179,1.205,1.227,1.245,1.261,1.276,
    1.289,1.302,1.314,1.326,1.338,1.351,1.363,1.375,1.387,1.399,
    1.411,1.424,1.437,1.450,1.465,1.484,1.511,1.554,1.613,1.823,
])
QMI_MAGNITUDE_PAPER = np.array([
    4.54,4.05,4.10,4.41,4.69,4.691,4.994,5.43,6.405,8.096,
    10.624,13.47,16.56,19.45,22.15,24.62,26.95,20.89,13.695,10.995,
    9.593,8.731,7.606,6.961,6.515,6.506,6.264,6.125,6.081,6.071,
    5.912,5.893,5.901,6.031,5.904,6.086,6.181,6.185,6.60,6.63,
    6.90,7.14,7.22,7.56,7.33,7.13,5.009,2.456,2.31,3.75,
])
QMI_PHASE_DEG_PAPER = np.array([
    176.8,152.8,147.6,146.4,149.6,157.4,169.5,-172.8,-152.0,-133.0,
    -116.5,-103.0,-88.8,-74.9,-59.5,-56.7,-21.8,-9.8,5.45,11.28,
    21.57,36.27,48.02,54.42,63.46,63.47,72.45,72.80,77.2,82.2,
    86.1,88.8,93.2,93.8,98.7,103.3,105.7,110.4,114.5,119.7,
    126.5,132.3,142.03,153.74,166.50,-172.15,-136.8,-126.8,-99.5,3.0,
])
QMI_PHASE_RAD_START = np.unwrap(np.deg2rad(QMI_PHASE_DEG_PAPER))
assert len(QMI_KNOTS_GEV) == len(QMI_MAGNITUDE_PAPER) == len(QMI_PHASE_RAD_START) == 50

## 3. Build the amplitude model

The $f_2(1270)$ coefficient is the fixed reference. The QMI contributes 100 floating dynamics parameters and the five other resonances contribute five complex coefficients, for 110 free parameters.

In [ ]:
def polar_coefficient(name, magnitude, phase_deg, *, fixed=False):
    phase = np.deg2rad(phase_deg)
    x = float(magnitude * np.cos(phase))
    y = float(magnitude * np.sin(phase))
    if fixed:
        return RealImag(x, y)
    return RealImag(
        Parameter.coefficient(f'{name}.x', x, owner=name, step=0.01),
        Parameter.coefficient(f'{name}.y', y, owner=name, step=0.01),
    )

S_OWNER = 'pipi_S_qmi'
s_magnitudes = tuple(
    Parameter.dynamics(f'S.mag_{i:02d}', float(v), owner=S_OWNER, bounds=(0.0, None), step=max(0.02, 0.01*float(v)))
    for i, v in enumerate(QMI_MAGNITUDE_PAPER)
)
s_phases = tuple(
    Parameter.dynamics(f'S.phase_{i:02d}', float(v), owner=S_OWNER, step=np.deg2rad(1.0))
    for i, v in enumerate(QMI_PHASE_RAD_START)
)
qmi_s_wave = QMI(tuple(QMI_KNOTS_GEV), s_magnitudes, s_phases, interpolation='natural')

coeff = {
    'rho770': polar_coefficient('rho770', 0.1201, 79.4),
    'omega782': polar_coefficient('omega782', 0.04001, -109.9),
    'rho1450': polar_coefficient('rho1450', 1.277, -115.2),
    'rho1700': polar_coefficient('rho1700', 0.873, -60.9),
    'f2_1270': polar_coefficient('f2_1270', 1.0, 0.0, fixed=True),
    'f2p_1525': polar_coefficient('f2p_1525', 0.1098, 178.1),
}

channel = DecayChannel('D_s+', ('pi-', 'pi+', 'pi+'))
angular = GooFitLegacyAngular()
bachelor_momentum_frame="parent"
R_D, R_R = 5.0, 1.5

def resonance(name, coefficient, mass, width, spin, *, lineshape=None):
    kwargs = {} if lineshape is None else {'lineshape': lineshape}
    return Resonance(
        name, (0, 1), coefficient, mass=mass, width=width, spin=spin,
        angular=angular, resonance_radius=R_R, parent_radius=R_D, bachelor_momentum_frame=bachelor_momentum_frame, **kwargs
    )

model = DecayModel(
    channel,
    [
        Resonance(S_OWNER, (0, 1), RealImag(1.0, 0.0), mass=1.0, width=0.0, spin=0,
                  lineshape=qmi_s_wave, angular=angular, resonance_radius=R_R, parent_radius=R_D, bachelor_momentum_frame=bachelor_momentum_frame),
        resonance('rho770', coeff['rho770'], 0.77526, 0.1491, 1, lineshape=GounarisSakurai()),
        resonance('omega782', coeff['omega782'], 0.78265, 0.00849, 1),
        resonance('rho1450', coeff['rho1450'], 1.465, 0.400, 1),
        resonance('rho1700', coeff['rho1700'], 1.720, 0.250, 1),
        resonance('f2_1270', coeff['f2_1270'], 1.2755, 0.1867, 2),
        resonance('f2p_1525', coeff['f2p_1525'], 1.5174, 0.086, 2),
    ],
    normalize_components=False,
)

print('components:', [c.name for c in model.components])
print('free model parameters:', sum(not p.fixed for p in model.parameters))
assert sum(not p.fixed for p in model.parameters) == 110

## 4. Read the full mass spectrum and efficiency

No Dalitz-background histogram is loaded. The background subtraction for the amplitude fit is entirely carried by the COW signal weights obtained from the mass fit below.

The mass-window mask is applied simultaneously to the discriminating variable and to all Dalitz coordinates, preserving event alignment.


In [ ]:
arrays = read_root_tree(
    DATA_FILE,
    DATA_TREE,
    {
        's12': S12_BRANCH,
        's13': S13_BRANCH,
        's23': S23_BRANCH,
        'mass': MASS_BRANCH,
    },
    cut=DATA_CUT,
    entry_start=ENTRY_START,
    entry_stop=ENTRY_STOP,
    library='np',
)

mass_all = np.asarray(arrays['mass'], dtype=float) * MASS_TO_GEV
mass_mask = (
    np.isfinite(mass_all)
    & (mass_all >= MASS_RANGE_GEV[0])
    & (mass_all <= MASS_RANGE_GEV[1])
)

mass = mass_all[mass_mask]
data = PhaseSpaceSample(
    s12=jnp.asarray(np.asarray(arrays['s12'])[mass_mask]),
    s13=jnp.asarray(np.asarray(arrays['s13'])[mass_mask]),
    s23=jnp.asarray(np.asarray(arrays['s23'])[mass_mask]),
    weights=jnp.ones(int(np.count_nonzero(mass_mask)), dtype=jnp.float64),
)

efficiency = histogram_efficiency_from_root(
    EFFICIENCY_FILE,
    EFFICIENCY_HIST,
    x_variable='s12',
    y_variable='s13',
)

print(f'Loaded {len(mass_all):,} candidates before the mass-range selection')
print(f'Using {data.size:,} candidates in {MASS_RANGE_GEV[0]:.3f} < m < {MASS_RANGE_GEV[1]:.3f} GeV')
for name in ('s12', 's13', 's23'):
    values = np.asarray(getattr(data, name))
    print(f'{name}: [{values.min():.6g}, {values.max():.6g}] GeV^2')


## 5. Extended mass fit — Chapter 4 model

The mass fit follows the analysis-note parameterisation:

\[
P_s(m)=f_{CB}\,CB(m;\alpha,n,\mu_{CB},\sigma_{CB})
+(1-f_{CB})\,G(m;\mu_G,\sigma_G),
\]

with

\[
\mu_G=\mu_{CB}+\mu_{\mathrm{offset}},
\qquad
\sigma_G=\sigma_{CB}\,\sigma_{\mathrm{Ratio}},
\]

and an exponential background. All signal-shape parameters, the exponential slope, and the two extended yields are free. The numerical starting values below reproduce the central values from Table 11 where available; the background slope is only a numerical seed because it is not quoted in that table.


In [ ]:
MASS_LOW, MASS_HIGH = MASS_RANGE_GEV


def _truncated_crystalball_pdf(x, mu_cb, sigma_cb, alpha, n):
    distribution = crystalball(alpha, n, loc=mu_cb, scale=sigma_cb)
    normalization = distribution.cdf(MASS_HIGH) - distribution.cdf(MASS_LOW)
    return distribution.pdf(x) / normalization


def _truncated_gaussian_pdf(x, mu_g, sigma_g):
    distribution = norm(loc=mu_g, scale=sigma_g)
    normalization = distribution.cdf(MASS_HIGH) - distribution.cdf(MASS_LOW)
    return distribution.pdf(x) / normalization


def signal_mass_pdf(
    x,
    alpha,
    n,
    mu_cb,
    sigma_cb,
    mu_offset,
    sigma_ratio,
    f_cb,
):
    mu_g = mu_cb + mu_offset
    sigma_g = sigma_cb * sigma_ratio

    cb = _truncated_crystalball_pdf(x, mu_cb, sigma_cb, alpha, n)
    gauss = _truncated_gaussian_pdf(x, mu_g, sigma_g)
    return f_cb * cb + (1.0 - f_cb) * gauss


def background_mass_pdf(x, slope):
    x = np.asarray(x, dtype=float)
    width = MASS_HIGH - MASS_LOW
    if abs(slope) < 1e-10:
        return np.ones_like(x) / width

    # Equivalent to exp(slope * x), but shifted for numerical stability.
    normalization = np.expm1(slope * width)
    return slope * np.exp(slope * (x - MASS_LOW)) / normalization


def extended_mass_density(
    x,
    nsig,
    nbkg,
    alpha,
    n,
    mu_cb,
    sigma_cb,
    mu_offset,
    sigma_ratio,
    f_cb,
    slope,
):
    signal = signal_mass_pdf(
        x,
        alpha,
        n,
        mu_cb,
        sigma_cb,
        mu_offset,
        sigma_ratio,
        f_cb,
    )
    background = background_mass_pdf(x, slope)
    return nsig + nbkg, nsig * signal + nbkg * background


mass_nll = ExtendedUnbinnedNLL(mass, extended_mass_density)

# Table 11 central values, converted from MeV to GeV where appropriate.
mass_fit = Minuit(
    mass_nll,
    nsig=771_130.0,
    nbkg=129_288.0,
    alpha=1.681,
    n=11.0,
    mu_cb=1.96978,
    sigma_cb=0.01039,
    mu_offset=0.00098,
    sigma_ratio=0.6666,
    f_cb=0.52,
    slope=-5.0,
)

mass_fit.limits['nsig'] = (0.0, None)
mass_fit.limits['nbkg'] = (0.0, None)
mass_fit.limits['alpha'] = (0.05, 10.0)
mass_fit.limits['n'] = (1.01, 100.0)
mass_fit.limits['mu_cb'] = MASS_RANGE_GEV
mass_fit.limits['sigma_cb'] = (5e-4, 0.05)
mass_fit.limits['mu_offset'] = (-0.02, 0.02)
mass_fit.limits['sigma_ratio'] = (0.05, 5.0)
mass_fit.limits['f_cb'] = (0.0, 1.0)
mass_fit.limits['slope'] = (-100.0, 100.0)

mass_fit.migrad()
mass_fit.hesse()
print(mass_fit)


In [ ]:
mass_parameters = mass_fit.values.to_dict()

mu_g_fit = mass_parameters['mu_cb'] + mass_parameters['mu_offset']
sigma_g_fit = mass_parameters['sigma_cb'] * mass_parameters['sigma_ratio']
sigma_eff_fit = np.sqrt(
    mass_parameters['f_cb'] * mass_parameters['sigma_cb']**2
    + (1.0 - mass_parameters['f_cb']) * sigma_g_fit**2
)

print(f'mu_G       = {1e3 * mu_g_fit:.3f} MeV')
print(f'sigma_G    = {1e3 * sigma_g_fit:.3f} MeV')
print(f'sigma_eff  = {1e3 * sigma_eff_fit:.3f} MeV')
print(
    'legacy signal region (mu_CB +/- 2 sigma_eff): '
    f'[{1e3 * (mass_parameters["mu_cb"] - 2 * sigma_eff_fit):.1f}, '
    f'{1e3 * (mass_parameters["mu_cb"] + 2 * sigma_eff_fit):.1f}] MeV'
)

x_mass = np.linspace(MASS_LOW, MASS_HIGH, 1000)
signal_curve = signal_mass_pdf(
    x_mass,
    mass_parameters['alpha'],
    mass_parameters['n'],
    mass_parameters['mu_cb'],
    mass_parameters['sigma_cb'],
    mass_parameters['mu_offset'],
    mass_parameters['sigma_ratio'],
    mass_parameters['f_cb'],
)
background_curve = background_mass_pdf(x_mass, mass_parameters['slope'])

fig, ax = plt.subplots(figsize=(8, 6))
counts, edges, _ = ax.hist(
    mass,
    bins=120,
    range=MASS_RANGE_GEV,
    histtype='step',
    label='data',
)
bin_width = edges[1] - edges[0]
ax.plot(
    x_mass,
    bin_width * (
        mass_parameters['nsig'] * signal_curve
        + mass_parameters['nbkg'] * background_curve
    ),
    label='extended fit',
)
ax.plot(
    x_mass,
    bin_width * mass_parameters['nsig'] * signal_curve,
    '--',
    label='signal',
)
ax.plot(
    x_mass,
    bin_width * mass_parameters['nbkg'] * background_curve,
    '--',
    label='background',
)
ax.set(xlabel=r'$m(\pi^-\pi^+\pi^+)$ [GeV]', ylabel='Candidates / bin')
ax.legend()
plt.show()


## 6. COW signal weights

The fitted signal and background shapes are frozen and passed to `sweights.Cow`. We choose the fitted total mass density as the COW variance function,

\[
I(m)=f_s\,g_s(m)+(1-f_s)\,g_b(m),
\]

which gives the minimum-variance two-component COW/sWeight construction for the fitted mass model. The resulting signal weights may be negative.

The COW weights are **not** stored in `PhaseSpaceSample.weights`: those weights have integration/proposal semantics in Jax-PWA. They are passed separately to `WeightedUnbinnedNLL`.


In [ ]:
def fitted_signal_mass_pdf(x):
    return signal_mass_pdf(
        x,
        mass_parameters['alpha'],
        mass_parameters['n'],
        mass_parameters['mu_cb'],
        mass_parameters['sigma_cb'],
        mass_parameters['mu_offset'],
        mass_parameters['sigma_ratio'],
        mass_parameters['f_cb'],
    )


def fitted_background_mass_pdf(x):
    return background_mass_pdf(x, mass_parameters['slope'])


fitted_signal_fraction = (
    mass_parameters['nsig']
    / (mass_parameters['nsig'] + mass_parameters['nbkg'])
)


def fitted_total_mass_pdf(x):
    return (
        fitted_signal_fraction * fitted_signal_mass_pdf(x)
        + (1.0 - fitted_signal_fraction) * fitted_background_mass_pdf(x)
    )


cow = Cow(
    MASS_RANGE_GEV,
    fitted_signal_mass_pdf,
    fitted_background_mass_pdf,
    Im=fitted_total_mass_pdf,
    renorm=False,
)

signal_weights = np.asarray(cow(mass), dtype=float)
background_weights = np.asarray(cow(mass, idx=1), dtype=float)

print(f'sum signal COW weights     = {signal_weights.sum():,.1f}')
print(f'fitted signal yield         = {mass_parameters["nsig"]:,.1f}')
print(f'sum background COW weights = {background_weights.sum():,.1f}')
print(f'fitted background yield     = {mass_parameters["nbkg"]:,.1f}')
print(f'min/max signal weight       = {signal_weights.min():.4g}, {signal_weights.max():.4g}')
print(f'negative signal weights     = {np.count_nonzero(signal_weights < 0):,}')
print(f'sum w^2                     = {np.sum(signal_weights**2):,.1f}')
print(
    'N_eff =',
    f'{signal_weights.sum()**2 / np.sum(signal_weights**2):,.1f}',
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
axes[0].scatter(mass, signal_weights, s=2, alpha=0.25)
axes[0].axhline(0.0, linewidth=1)
axes[0].set(
    xlabel=r'$m(\pi^-\pi^+\pi^+)$ [GeV]',
    ylabel='signal COW weight',
)
axes[1].hist(
    mass,
    bins=120,
    range=MASS_RANGE_GEV,
    weights=signal_weights,
    histtype='step',
    label='signal-weighted data',
)
axes[1].plot(
    x_mass,
    bin_width * signal_weights.sum() * fitted_signal_mass_pdf(x_mass),
    label='fitted signal shape',
)
axes[1].set(
    xlabel=r'$m(\pi^-\pi^+\pi^+)$ [GeV]',
    ylabel='Weighted candidates / bin',
)
axes[1].legend()
plt.show()


## 7. Pre-fit signal-model / COW-weighted-data comparison

The old signal+background toy comparison is replaced by a signal-only toy compared with the COW-weighted data. Histogram uncertainties use $\sqrt{\sum w_i^2}$ in each bin.


In [ ]:
published_start = {p.name: float(p.value) for p in model.parameters if not p.fixed}


def plot_weighted_projection_comparison(variable, generated, bins=60):
    observed = np.asarray(getattr(data, variable))
    generated_values = np.asarray(getattr(generated, variable))
    edges = np.linspace(
        min(observed.min(), generated_values.min()),
        max(observed.max(), generated_values.max()),
        bins + 1,
    )

    weighted_counts, _ = np.histogram(
        observed,
        bins=edges,
        weights=signal_weights,
    )
    weighted_variance, _ = np.histogram(
        observed,
        bins=edges,
        weights=signal_weights**2,
    )
    generated_counts, _ = np.histogram(generated_values, bins=edges)
    generated_counts = (
        generated_counts
        * signal_weights.sum()
        / max(generated_counts.sum(), 1)
    )

    centers = 0.5 * (edges[:-1] + edges[1:])
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.errorbar(
        centers,
        weighted_counts,
        yerr=np.sqrt(weighted_variance),
        fmt='o',
        markersize=3,
        label='COW-weighted data',
    )
    ax.stairs(generated_counts, edges, label='prefit signal toy')
    ax.set(
        xlabel='$' + variable + '$ [GeV$^2$]',
        ylabel='Weighted candidates',
    )
    ax.legend()
    plt.show()


if RUN_PREFIT_TOY_COMPARISON:
    prefit_toy = generate_toy(
        model,
        max(1, int(round(signal_weights.sum()))),
        parameters=published_start,
        efficiency=efficiency,
        seed=TOY_SEED,
    )

    for variable in ('s12', 's13', 's23'):
        plot_weighted_projection_comparison(variable, prefit_toy)


## 8. Weighted Dalitz fit

The `FitSession` is now signal-only. `WeightedUnbinnedNLL` is shown explicitly below; `session.fit(weights=..., covariance="sweight")` builds the same weighted objective internally and replaces the returned Minuit covariance by the $H_w^{-1}H_{w^2}H_w^{-1}$ correction.

There is no `BackgroundSpec`, no sideband histogram, and no fixed signal fraction in the Dalitz likelihood.


In [ ]:
session = FitSession(
    model,
    data,
    efficiency=efficiency,
)

weighted_nll = WeightedUnbinnedNLL(
    session._cached_signal_logpdf,
    data.as_dict(),
    jnp.asarray(signal_weights),
)

start = dict(published_start)
print('Weighted NLL at published start:', float(weighted_nll(start)))

if RUN_FIT:
    result = session.fit(
        start,
        weights=jnp.asarray(signal_weights),
        covariance='sweight',
        strategy=1,
        hesse=False,
        simplex=False,
        hessian='jax',
        hessian_batch_size=1,
        ncall=250_000,
        tolerance=1e-4,
        verbose=3,
    )


## 9. Weighted-fit diagnostics

The ordinary `session.plot_projection()` currently displays unweighted data, so the diagnostic projections below explicitly histogram the data with the COW weights and use $\sqrt{\sum w_i^2}$ errors. The smooth model projection is generated from the fitted signal model and scaled to $\sum_i w_i$.


In [ ]:
def plot_weighted_fit_projection(result, variable, bins=60, projection_size=500_000):
    values = session.result_values(result)
    observed = np.asarray(getattr(data, variable))
    edges = np.linspace(observed.min(), observed.max(), bins + 1)

    weighted_counts, _ = np.histogram(
        observed,
        bins=edges,
        weights=signal_weights,
    )
    weighted_variance, _ = np.histogram(
        observed,
        bins=edges,
        weights=signal_weights**2,
    )

    projection_sample = session._get_projection_sample(projection_size, TOY_SEED)
    total = np.zeros(bins, dtype=float)
    components = []
    for name, component_sample, component_weights in session._projection_components(
        values,
        projection_sample,
    ):
        component_values = np.asarray(getattr(component_sample, variable))
        counts, _ = np.histogram(
            component_values,
            bins=edges,
            weights=np.asarray(component_weights),
        )
        total += counts
        components.append((name, counts))

    scale = signal_weights.sum() / max(total.sum(), 1e-300)
    centers = 0.5 * (edges[:-1] + edges[1:])

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.errorbar(
        centers,
        weighted_counts,
        yerr=np.sqrt(weighted_variance),
        fmt='o',
        markersize=3,
        label='COW-weighted data',
    )
    for name, counts in components:
        ax.stairs(scale * counts, edges, label=name, alpha=0.65)
    ax.stairs(scale * total, edges, label='total fit', linewidth=2)
    ax.set(
        xlabel='$' + variable + '$ [GeV$^2$]',
        ylabel='Weighted candidates',
    )
    ax.legend(fontsize='small')
    plt.show()


if RUN_FIT:
    values = session.result_values(result)
    print('Weighted NLL fit:', float(weighted_nll(values)))
    session.print_fit_fractions(
        result,
        acceptance_weighted=False,
        include_interference=True,
        precision=4,
    )

    fit_mag = np.array([values[f'S.mag_{i:02d}'] for i in range(50)])
    fit_phase = np.array([values[f'S.phase_{i:02d}'] for i in range(50)])
    paper_complex = QMI_MAGNITUDE_PAPER * np.exp(1j * QMI_PHASE_RAD_START)
    fit_complex = fit_mag * np.exp(1j * fit_phase)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
    axes[0].plot(QMI_KNOTS_GEV, QMI_MAGNITUDE_PAPER, 'o', label='paper start')
    axes[0].plot(QMI_KNOTS_GEV, fit_mag, '-', label='weighted fit')
    axes[0].set(xlabel=r'$m(\pi^+\pi^-)$ [GeV]', ylabel='magnitude')
    axes[0].legend()

    axes[1].plot(
        QMI_KNOTS_GEV,
        np.rad2deg(QMI_PHASE_RAD_START),
        'o',
        label='paper start',
    )
    axes[1].plot(
        QMI_KNOTS_GEV,
        np.rad2deg(fit_phase),
        '-',
        label='weighted fit',
    )
    axes[1].set(xlabel=r'$m(\pi^+\pi^-)$ [GeV]', ylabel='phase [deg]')
    axes[1].legend()

    axes[2].plot(
        paper_complex.real,
        paper_complex.imag,
        'o',
        label='paper start',
    )
    axes[2].plot(
        fit_complex.real,
        fit_complex.imag,
        '-',
        label='weighted fit',
    )
    axes[2].set(xlabel=r'Re $A_S$', ylabel=r'Im $A_S$')
    axes[2].legend()
    plt.show()

    for variable in ('s12', 's13', 's23'):
        plot_weighted_fit_projection(result, variable, bins=60)
